# Chapter 15: Absolute Geometry

Source span inspected: Coxeter, *Introduction to Geometry*, chapter 15, printed pages 263-286 (PDF pages 281-304). The PDF pages are scanned, so the source was inspected from temporary rendered page images rather than copied text. This notebook is original teaching material: it does not reproduce textbook prose, exercises, figures, screenshots, page crops, or layouts.

## Chapter Goal

Absolute geometry asks how much geometry can be carried by order, congruence, reflection, and incidence before Euclid's parallel postulate is used. The computational goal here is to turn that question into inspectable invariants: distances preserved by congruence, ideal endpoints that explain parallelism, products of reflections that make isometries, group orders constrained by pole counts, crystallographic periods forced by lattice traces, spherical triangles that drive polyhedral kaleidoscopes, and inversion mirrors whose angle sums separate finite, Euclidean, and hyperbolic behavior.

## Computational Translation Guide

| Source idea | Computational representation | What to inspect |
| --- | --- | --- |
| Congruence axioms | segment transfer, triangle side checks, additive length data | equality survives relocation and composition |
| Parallel rays and ultraparallels | Poincare disk model geodesics as a consistency model for absolute claims | shared ideal endpoint versus common perpendicular |
| Isometry | matrices for half-turns, translations, and reflections | products preserve distance and orientation parity |
| Finite rotation groups | pole signatures `(p1, p2, p3)` and concrete rotation matrices | the equation `1/p1 + 1/p2 + 1/p3 = 1 + 2/|G|` |
| Finite isometry groups | index-two rotation subgroups and central inversion extensions | direct and opposite isometries split into equal counts |
| Geometrical crystallography | lattice rotation trace `2 cos(2 pi/n)` | only periods 1, 2, 3, 4, and 6 can preserve a lattice |
| Polyhedral kaleidoscopes | spherical reflection chambers | a fundamental spherical triangle tiles the sphere |
| Inversion-generated groups | circle inversions and orbit growth | angle sum controls finite, Euclidean, or infinite behavior |

Library choices follow the course catalog: Matplotlib for durable 2D proof diagrams, NetworkX for the proof dependency graph, SymPy/Fractions for exact group and lattice checks, Trimesh for polyhedral mesh axes, and Plotly for rotatable 3D rotation-axis and spherical-kaleidoscope artifacts.


In [ ]:
from pathlib import Path
import sys, json, csv, math, itertools
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import sympy as sp
import plotly.graph_objects as go
import trimesh
from IPython.display import Markdown, display

def find_book_root() -> Path:
    here = Path.cwd().resolve()
    candidates = []
    for base in [here, *here.parents]:
        candidates.append(base)
        candidates.append(base / "Introduction-to-Geometry")
    for candidate in candidates:
        if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
            return candidate
    raise RuntimeError("Could not locate Introduction-to-Geometry book root")

BOOK_ROOT = find_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from utils.artifacts import assert_artifacts, display_artifact

CHAPTER_NO = 15
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-15"
DIRS = {kind: ARTIFACT_ROOT / kind for kind in ["figures", "html", "checks", "tables", "data"]}
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

GENERATED_ARTIFACTS = []
CHECKS = {"source_span": {"printed_pages": "263-286", "pdf_pages": "281-304"}, "chapter": CHAPTER_NO}
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def record(path: Path) -> Path:
    path = Path(path)
    if path not in GENERATED_ARTIFACTS:
        GENERATED_ARTIFACTS.append(path)
    return path

def book_rel(path: Path) -> str:
    return Path(path).resolve().relative_to(BOOK_ROOT.resolve()).as_posix()

def write_json_artifact(kind: str, filename: str, payload) -> Path:
    path = DIRS[kind] / filename
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    return record(path)

def write_csv_artifact(filename: str, rows) -> Path:
    rows = list(rows)
    path = DIRS["tables"] / filename
    with path.open("w", newline="", encoding="utf-8") as handle:
        if rows:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
            writer.writeheader()
            writer.writerows(rows)
    return record(path)

def save_figure(filename: str, fig, dpi: int = 180) -> Path:
    path = DIRS["figures"] / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return record(path)

def write_html(filename: str, fig) -> Path:
    path = DIRS["html"] / filename
    fig.write_html(str(path), include_plotlyjs=True, full_html=True)
    return record(path)

def show_artifact(path: Path, width: int = 760):
    display(Markdown(f"**Artifact:** `{book_rel(path)}`"))
    display_artifact(path, width=width)

def markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = ["| " + " | ".join(str(row.get(col, "")) for col in columns) + " |" for row in rows]
    return "\n".join([header, sep, *body])

def unit(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)
    if n == 0:
        raise ValueError("zero vector")
    return v / n

def dist(a, b):
    return float(np.linalg.norm(np.asarray(a, dtype=float) - np.asarray(b, dtype=float)))

display(Markdown(f"Working in `{BOOK_ROOT.name}` with artifacts under `{book_rel(ARTIFACT_ROOT)}`."))


## Chapter Storyboard

The chapter is organized as a chain of increasingly structured invariants. The first two sections are deliberately small: congruence and parallelism are not just words but equivalence relations whose proof burden can be drawn. The middle sections move from individual isometries to finite groups, where counting arguments replace metric measurements. The last sections show two ways group theory becomes geometry again: crystallographic restrictions in Euclidean lattices, and reflection or inversion chambers that tile the sphere or disk.

The proof-dependency graph below is not a replacement for proof. It is a map of which hypotheses carry which later conclusions. Read arrows as "is used by" rather than as chronological page order.


In [ ]:
proof_edges = [
    ("ordered geometry", "parallel symmetry"),
    ("ordered geometry", "parallel transitivity"),
    ("congruence axioms", "length and angle"),
    ("length and angle", "reflection construction"),
    ("reflection construction", "isometry products"),
    ("parallel transitivity", "pencils at infinity"),
    ("isometry products", "finite fixed point"),
    ("finite fixed point", "rotation pole equation"),
    ("rotation pole equation", "C_n, D_n, A4, S4, A5"),
    ("C_n, D_n, A4, S4, A5", "finite isometry groups"),
    ("finite isometry groups", "crystal point groups"),
    ("reflection construction", "polyhedral kaleidoscopes"),
    ("polyhedral kaleidoscopes", "inversion chambers"),
]
G = nx.DiGraph(proof_edges)
assert nx.is_directed_acyclic_graph(G)
assert nx.has_path(G, "congruence axioms", "finite isometry groups")
assert nx.has_path(G, "reflection construction", "inversion chambers")

fig, ax = plt.subplots(figsize=(10, 6))
pos = nx.spring_layout(G, seed=15, k=0.9)
levels = {
    "ordered geometry": 0, "congruence axioms": 0, "length and angle": 1,
    "parallel symmetry": 1, "parallel transitivity": 1, "reflection construction": 2,
    "pencils at infinity": 2, "isometry products": 3, "finite fixed point": 4,
    "rotation pole equation": 5, "C_n, D_n, A4, S4, A5": 6,
    "finite isometry groups": 7, "crystal point groups": 8,
    "polyhedral kaleidoscopes": 4, "inversion chambers": 5,
}
for node, level in levels.items():
    if node in pos:
        pos[node][0] = level
node_colors = ["#f2c14e" if G.in_degree(n) == 0 else "#4b9aaa" if G.out_degree(n) else "#d95d39" for n in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, width=1.2, edge_color="#566573")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=1500, edgecolors="#1f2933", linewidths=0.8)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color="#111827")
ax.set_title("Absolute geometry proof dependencies", loc="left", fontsize=14, weight="bold")
ax.axis("off")
proof_path = save_figure("proof-dependency-graph.png", fig)
CHECKS["proof_dependency_graph"] = {
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "acyclic": nx.is_directed_acyclic_graph(G),
    "congruence_to_finite_isometry": nx.has_path(G, "congruence axioms", "finite isometry groups"),
}
show_artifact(proof_path)


## 1. Congruence as Transfer, Addition, and Triangle Control

The source introduces congruence as a primitive equivalence relation for segments and then extends it to angles and triangles. Computationally, congruence must survive relocation: a segment can be copied to any ray, adjacent congruent pieces add to congruent wholes, and a triangle carried by a rigid motion keeps all side lengths and angles.


In [ ]:
A = np.array([0.0, 0.0]); B = np.array([2.4, 0.0])
C = np.array([0.25, 1.25])
ray_dir = unit([math.cos(math.radians(27)), math.sin(math.radians(27))])
D = C + dist(A, B) * ray_dir
P0, P1, P2 = np.array([0.0, 0.0]), np.array([1.1, 0.0]), np.array([2.8, 0.0])
Q0, Q1, Q2 = np.array([0.0, -0.55]), np.array([1.1, -0.55]), np.array([2.8, -0.55])
T = np.array([[0.0, 0.0], [1.6, 0.2], [0.55, 1.25]])
theta = math.radians(34)
R = np.array([[math.cos(theta), -math.sin(theta)], [math.sin(theta), math.cos(theta)]])
T2 = T @ R.T + np.array([3.25, 0.45])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
ax = axes[0]
ax.plot([A[0], B[0]], [A[1], B[1]], color="#1f77b4", lw=3, label="source segment")
ax.plot([C[0], D[0]], [C[1], D[1]], color="#d95d39", lw=3, label="copied to ray")
ax.scatter([A[0], B[0], C[0], D[0]], [A[1], B[1], C[1], D[1]], color="#111827", s=25)
for label, point in [("A", A), ("B", B), ("C", C), ("D", D)]:
    ax.text(point[0] + 0.04, point[1] + 0.04, label)
ax.set_title("15.11: copy a segment"); ax.legend(loc="lower right", fontsize=8)

ax = axes[1]
ax.plot([P0[0], P2[0]], [P0[1], P2[1]], color="#333333", lw=2)
ax.plot([Q0[0], Q2[0]], [Q0[1], Q2[1]], color="#333333", lw=2)
ax.scatter([P0[0], P1[0], P2[0], Q0[0], Q1[0], Q2[0]], [P0[1], P1[1], P2[1], Q0[1], Q1[1], Q2[1]], color="#111827", s=20)
for label, point in [("A", P0), ("B", P1), ("C", P2), ("A'", Q0), ("B'", Q1), ("C'", Q2)]:
    ax.text(point[0], point[1] + 0.08, label, ha="center")
ax.annotate("AB = A'B'", xy=((P0[0]+P1[0])/2, 0.12), ha="center")
ax.annotate("BC = B'C'", xy=((P1[0]+P2[0])/2, 0.12), ha="center")
ax.annotate("so AC = A'C'", xy=(1.4, -0.9), ha="center", color="#0f766e")
ax.set_title("15.14: additive length")

ax = axes[2]
for tri, color, label in [(T, "#1f77b4", "triangle"), (T2, "#d95d39", "rigid image")]:
    closed = np.vstack([tri, tri[0]])
    ax.plot(closed[:, 0], closed[:, 1], color=color, lw=2.4, label=label)
    ax.scatter(tri[:, 0], tri[:, 1], color=color, s=20)
ax.set_title("15.15: SSS controls triangles"); ax.legend(loc="upper right", fontsize=8)
for ax in axes:
    ax.set_aspect("equal", adjustable="box"); ax.grid(True, color="#e5e7eb", lw=0.6); ax.set_xticks([]); ax.set_yticks([])
congruence_path = save_figure("congruence-invariant-tracker.png", fig)

side_errors = [abs(dist(T[i], T[j]) - dist(T2[i], T2[j])) for i, j in [(0, 1), (1, 2), (2, 0)]]
CHECKS["congruence"] = {
    "segment_transfer_error": abs(dist(A, B) - dist(C, D)),
    "additive_length_error": abs(dist(P0, P2) - (dist(P0, P1) + dist(P1, P2))) + abs(dist(Q0, Q2) - (dist(Q0, Q1) + dist(Q1, Q2))),
    "max_triangle_side_error": max(side_errors),
}
assert CHECKS["congruence"]["segment_transfer_error"] < 1e-12
assert CHECKS["congruence"]["additive_length_error"] < 1e-12
assert CHECKS["congruence"]["max_triangle_side_error"] < 1e-12
show_artifact(congruence_path)
CHECKS["congruence"]


## 2. Parallelism Without Assuming Euclid's Fifth Postulate

The chapter treats parallelism as a relation on rays that becomes symmetric and transitive inside absolute geometry. A Euclidean picture hides why this matters. The Poincare disk below is used only as a consistency model: parallel rays share an ideal endpoint and meet at angle zero there; ultraparallel lines share no endpoint and admit a common perpendicular.


In [ ]:
def boundary_point(angle):
    return np.array([math.cos(angle), math.sin(angle)], dtype=float)

def orthogonal_circle_for_boundary_angles(a0, a1):
    p = boundary_point(a0); q = boundary_point(a1)
    if abs(np.dot(p, q) + 1.0) < 1e-10:
        return {"kind": "diameter", "p": p, "q": q}
    center = np.linalg.solve(np.vstack([p, q]), np.ones(2))
    radius = math.sqrt(np.dot(center, center) - 1.0)
    return {"kind": "circle", "p": p, "q": q, "center": center, "radius": radius}

def geodesic_points(geo, samples=240):
    if geo["kind"] == "diameter":
        t = np.linspace(-1, 1, samples); d = unit(geo["p"])
        return np.outer(t, d)
    c = geo["center"]; r = geo["radius"]
    a0 = math.atan2(geo["p"][1] - c[1], geo["p"][0] - c[0])
    a1 = math.atan2(geo["q"][1] - c[1], geo["q"][0] - c[0])
    def arc(delta):
        angles = a0 + np.linspace(0, delta, samples)
        return c + r * np.column_stack([np.cos(angles), np.sin(angles)])
    d_ccw = (a1 - a0) % (2 * math.pi)
    candidates = [arc(d_ccw), arc(d_ccw - 2 * math.pi)]
    valid = [pts for pts in candidates if np.max(np.linalg.norm(pts, axis=1)) <= 1.0001]
    return min(valid or candidates, key=lambda pts: np.mean(np.linalg.norm(pts, axis=1)))

def tangent_at_endpoint(geo, endpoint):
    endpoint = unit(endpoint)
    if geo["kind"] == "diameter":
        return endpoint
    normal = endpoint - geo["center"]
    tangent = np.array([-normal[1], normal[0]])
    if np.dot(tangent, endpoint) < 0:
        tangent = -tangent
    return unit(tangent)

def angle_between(u, v):
    return math.acos(max(-1.0, min(1.0, abs(float(np.dot(unit(u), unit(v)))))))

p = orthogonal_circle_for_boundary_angles(0, math.pi)
q = orthogonal_circle_for_boundary_angles(0, math.radians(125))
r = orthogonal_circle_for_boundary_angles(math.radians(58), math.radians(122))
common_perp = orthogonal_circle_for_boundary_angles(math.pi / 2, -math.pi / 2)
fig, ax = plt.subplots(figsize=(7, 7))
t = np.linspace(0, 2 * math.pi, 500)
ax.plot(np.cos(t), np.sin(t), color="#111827", lw=1.8, label="ideal boundary")
for geo, color, label, lw in [
    (p, "#1f77b4", "p: reference geodesic", 2.6),
    (q, "#d95d39", "q: shares one ideal end", 2.6),
    (r, "#0f766e", "r: ultraparallel to p", 2.6),
    (common_perp, "#7c3aed", "common perpendicular", 1.8),
]:
    pts = geodesic_points(geo)
    ax.plot(pts[:, 0], pts[:, 1], color=color, lw=lw, label=label)
ax.scatter([1], [0], color="#d95d39", s=45, zorder=5)
ax.text(1.03, 0.02, "shared end", color="#d95d39")
ax.text(-0.8, -0.12, "parallel rays", color="#1f77b4")
ax.text(-0.18, 0.7, "common perpendicular", color="#7c3aed", rotation=90, ha="center")
ax.text(-0.55, 0.47, "ultraparallel", color="#0f766e")
ax.set_aspect("equal"); ax.set_xlim(-1.08, 1.08); ax.set_ylim(-1.08, 1.08)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("Parallel and ultraparallel geodesics in a consistency model", loc="left", weight="bold")
ax.legend(loc="lower left", fontsize=8)
parallel_path = save_figure("parallel-pencils-and-ultraparallels.png", fig)
shared_endpoint_angle = angle_between(tangent_at_endpoint(p, np.array([1.0, 0.0])), tangent_at_endpoint(q, np.array([1.0, 0.0])))
r_pts = geodesic_points(r); common_perp_hits_r = r_pts[np.argmin(r_pts[:, 1])]
CHECKS["parallelism"] = {
    "shared_ideal_endpoint_tangent_angle": shared_endpoint_angle,
    "common_perpendicular_x_error": abs(float(common_perp_hits_r[0])),
    "max_geodesic_norm": max(float(np.max(np.linalg.norm(geodesic_points(g), axis=1))) for g in [p, q, r, common_perp]),
}
assert shared_endpoint_angle < 1e-10
assert CHECKS["parallelism"]["common_perpendicular_x_error"] < 1e-2
assert CHECKS["parallelism"]["max_geodesic_norm"] <= 1.001
show_artifact(parallel_path)
CHECKS["parallelism"]


## 3. Isometries as Products of Half-Turns and Reflections

The isometry section separates motions with invariant points from translations and parallel displacements. A half-turn about `O` followed by a half-turn about `O'` is translation by `2(O' - O)`. Two reflections in parallel lines translate by twice the separation. Donkin's triangle identity says that the three directed translations along a triangle's sides, each taken with twice the side vector, close up to the identity.


In [ ]:
def half_turn(center, x):
    return 2 * np.asarray(center, dtype=float) - np.asarray(x, dtype=float)

def reflect_horizontal(y0, x):
    x = np.asarray(x, dtype=float)
    return np.array([x[0], 2 * y0 - x[1]])

O = np.array([0.0, 0.0]); Op = np.array([1.25, 0.45])
probe_points = np.array([[-0.8, -0.25], [0.2, 0.8], [1.4, -0.1], [0.7, -0.9]])
product_half_turns = np.array([half_turn(Op, half_turn(O, x)) for x in probe_points])
expected_half_turns = probe_points + 2 * (Op - O)
half_turn_error = float(np.max(np.linalg.norm(product_half_turns - expected_half_turns, axis=1)))
y_a, y_b = -0.35, 0.55
product_reflections = np.array([reflect_horizontal(y_b, reflect_horizontal(y_a, x)) for x in probe_points])
expected_reflections = probe_points + np.array([0.0, 2 * (y_b - y_a)])
parallel_reflection_error = float(np.max(np.linalg.norm(product_reflections - expected_reflections, axis=1)))
Tri = np.array([[0.0, 0.0], [1.7, 0.25], [0.55, 1.35]])
translations = [2 * (Tri[1] - Tri[0]), 2 * (Tri[2] - Tri[1]), 2 * (Tri[0] - Tri[2])]
donkin_vector = np.sum(translations, axis=0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
ax = axes[0]
ax.scatter(probe_points[:, 0], probe_points[:, 1], color="#1f77b4", label="before")
ax.scatter(product_half_turns[:, 0], product_half_turns[:, 1], color="#d95d39", label="after")
for x, y in zip(probe_points, product_half_turns):
    ax.arrow(x[0], x[1], y[0] - x[0], y[1] - x[1], head_width=0.05, length_includes_head=True, color="#566573", alpha=0.7)
ax.scatter([O[0], Op[0]], [O[1], Op[1]], color="#111827", s=40)
ax.text(O[0] - 0.08, O[1] - 0.12, "O"); ax.text(Op[0] + 0.04, Op[1], "O'")
ax.set_title("two half-turns = translation"); ax.legend(fontsize=8)
ax = axes[1]
ax.axhline(y_a, color="#1f77b4", lw=2, label="mirror a"); ax.axhline(y_b, color="#d95d39", lw=2, label="mirror b")
ax.scatter(probe_points[:, 0], probe_points[:, 1], color="#1f77b4"); ax.scatter(product_reflections[:, 0], product_reflections[:, 1], color="#d95d39")
for x, y in zip(probe_points, product_reflections):
    ax.arrow(x[0], x[1], 0, y[1] - x[1], head_width=0.04, length_includes_head=True, color="#566573", alpha=0.7)
ax.set_title("parallel reflections = translation"); ax.legend(fontsize=8)
ax = axes[2]
closed = np.vstack([Tri, Tri[0]]); ax.plot(closed[:, 0], closed[:, 1], color="#111827", lw=2)
origin = np.array([2.4, 0.2]); running = origin.copy()
for vec, color, label in zip(translations, ["#1f77b4", "#d95d39", "#0f766e"], ["2AB", "2BC", "2CA"]):
    ax.arrow(running[0], running[1], vec[0], vec[1], head_width=0.06, length_includes_head=True, color=color, lw=2)
    ax.text(*(running + 0.5 * vec + np.array([0.02, 0.02])), label, color=color)
    running = running + vec
ax.scatter([origin[0]], [origin[1]], color="#111827", s=25); ax.set_title("Donkin translation loop")
for ax in axes:
    ax.set_aspect("equal", adjustable="box"); ax.grid(True, color="#e5e7eb", lw=0.6); ax.set_xticks([]); ax.set_yticks([])
isometry_path = save_figure("reflection-products-isometry-checks.png", fig)
CHECKS["isometry_products"] = {
    "half_turn_product_error": half_turn_error,
    "parallel_reflection_product_error": parallel_reflection_error,
    "donkin_loop_norm": float(np.linalg.norm(donkin_vector)),
}
assert half_turn_error < 1e-12
assert parallel_reflection_error < 1e-12
assert np.linalg.norm(donkin_vector) < 1e-12
show_artifact(isometry_path)
CHECKS["isometry_products"]


## 4. Finite Rotation Groups

A finite group of isometries has an invariant point: average any orbit, or equivalently take the center of the smallest enclosing sphere. Once all rotations act on a sphere around that point, the group is constrained by its poles. Except for cyclic groups, the possible signatures are `(2, 2, p)`, `(2, 3, 3)`, `(2, 3, 4)`, and `(2, 3, 5)`, giving the dihedral, tetrahedral, octahedral, and icosahedral rotation groups.


In [ ]:
def tetra_rotation_matrices():
    vertices = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]], dtype=float).T
    gram_inv = np.linalg.inv(vertices @ vertices.T)
    rotations = []
    for perm in itertools.permutations(range(4)):
        target = vertices[:, perm]
        Rm = target @ vertices.T @ gram_inv
        if np.max(np.abs(Rm @ vertices - target)) < 1e-10 and np.linalg.det(Rm) > 0.999:
            rotations.append(Rm)
    return rotations

def signed_permutation_matrices(det_value=None):
    matrices = []
    for perm in itertools.permutations(range(3)):
        for signs in itertools.product([-1, 1], repeat=3):
            M = np.zeros((3, 3))
            for row, col in enumerate(perm):
                M[row, col] = signs[row]
            det = round(np.linalg.det(M))
            if det_value is None or det == det_value:
                matrices.append(M)
    return matrices

def canonical_axis(v, decimals=8):
    v = unit(v)
    for component in v:
        if abs(component) > 1e-9:
            if component < 0:
                v = -v
            break
    return tuple(np.round(v, decimals))

rotation_rows = [
    {"group": "C_n", "sample_order": 5, "pole_signature": "two n-gonal pole sets", "source_role": "single rotation axis"},
    {"group": "D_p", "sample_order": 10, "pole_signature": "(2, 2, 5)", "source_role": "prism/pyramid rotations"},
    {"group": "A4", "sample_order": 12, "pole_signature": "(2, 3, 3)", "source_role": "tetrahedron rotations"},
    {"group": "S4", "sample_order": 24, "pole_signature": "(2, 3, 4)", "source_role": "cube/octahedron rotations"},
    {"group": "A5", "sample_order": 60, "pole_signature": "(2, 3, 5)", "source_role": "dodecahedron/icosahedron rotations"},
]
rotation_table_path = write_csv_artifact("finite-rotation-groups.csv", rotation_rows)
signature_checks = {"D_5": (2, 2, 5, 10), "A4": (2, 3, 3, 12), "S4": (2, 3, 4, 24), "A5": (2, 3, 5, 60)}
for name, (p1, p2, p3, order) in signature_checks.items():
    assert Fraction(1, p1) + Fraction(1, p2) + Fraction(1, p3) == Fraction(1, 1) + Fraction(2, order), name
tetra_rots = tetra_rotation_matrices(); octa_rots = signed_permutation_matrices(det_value=1)
probe = np.array([0.23, -0.41, 0.89])
tetra_orbit = np.array([R @ probe for R in tetra_rots]); tetra_centroid = tetra_orbit.mean(axis=0)
centroid_errors = [np.linalg.norm(R @ tetra_centroid - tetra_centroid) for R in tetra_rots]
fig, ax = plt.subplots(figsize=(8, 4.5))
groups = [row["group"] for row in rotation_rows]; orders = [row["sample_order"] for row in rotation_rows]
ax.bar(groups, orders, color=["#4b9aaa", "#4b9aaa", "#d95d39", "#d95d39", "#d95d39"])
for x, order, row in zip(groups, orders, rotation_rows):
    ax.text(x, order + 1.5, row["pole_signature"], ha="center", fontsize=8)
ax.set_ylabel("sample group order"); ax.set_title("Finite rotation groups from pole signatures", loc="left", weight="bold")
ax.grid(axis="y", color="#e5e7eb")
rotation_fig_path = save_figure("finite-rotation-classification.png", fig)
CHECKS["finite_rotations"] = {
    "tetra_rotation_count": len(tetra_rots),
    "octa_rotation_count": len(octa_rots),
    "signature_equations_checked": sorted(signature_checks),
    "tetra_orbit_centroid_norm": float(np.linalg.norm(tetra_centroid)),
    "tetra_centroid_invariance_error": float(max(centroid_errors)),
}
assert len(tetra_rots) == 12
assert len(octa_rots) == 24
assert max(centroid_errors) < 1e-14
display(Markdown(markdown_table(rotation_rows, ["group", "sample_order", "pole_signature", "source_role"])))
show_artifact(rotation_fig_path)
display(Markdown(f"Rotation table written to `{book_rel(rotation_table_path)}`."))
CHECKS["finite_rotations"]


In [ ]:
def unique_axes_from_vectors(vectors):
    return sorted({canonical_axis(v) for v in vectors})

tetra_vertices = np.array([[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]], dtype=float)
tetra_c3_axes = unique_axes_from_vectors(tetra_vertices)
tetra_c2_axes = unique_axes_from_vectors([tetra_vertices[0] + tetra_vertices[1], tetra_vertices[0] + tetra_vertices[2], tetra_vertices[0] + tetra_vertices[3]])
ico = trimesh.creation.icosphere(subdivisions=0, radius=1.0)
ico_vertices = np.asarray(ico.vertices); ico_faces = np.asarray(ico.faces); ico_edges = np.asarray(ico.edges_unique)
ico_c5_axes = unique_axes_from_vectors(ico_vertices)
ico_c3_axes = unique_axes_from_vectors([ico_vertices[face].mean(axis=0) for face in ico_faces])
ico_c2_axes = unique_axes_from_vectors([ico_vertices[edge].mean(axis=0) for edge in ico_edges])

def axis_trace(axes, name, color, width=4):
    xs, ys, zs = [], [], []
    for raw in axes:
        d = unit(np.asarray(raw, dtype=float))
        xs += [-d[0], d[0], None]; ys += [-d[1], d[1], None]; zs += [-d[2], d[2], None]
    return go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=color, width=width), name=name)

u = np.linspace(0, 2 * math.pi, 60); v = np.linspace(0, math.pi, 30)
xs = np.outer(np.cos(u), np.sin(v)); ys = np.outer(np.sin(u), np.sin(v)); zs = np.outer(np.ones_like(u), np.cos(v))
fig = go.Figure()
fig.add_trace(go.Surface(x=xs, y=ys, z=zs, opacity=0.14, showscale=False, colorscale=[[0, "#dbeafe"], [1, "#dbeafe"]], name="unit sphere"))
fig.add_trace(axis_trace(tetra_c3_axes, "tetra A4: 4 threefold axes", "#d95d39", 5))
fig.add_trace(axis_trace(tetra_c2_axes, "tetra A4: 3 half-turn axes", "#f59e0b", 5))
fig.add_trace(axis_trace(ico_c5_axes, "icosa A5: 6 fivefold axes", "#1f77b4", 3))
fig.add_trace(axis_trace(ico_c3_axes, "icosa A5: 10 threefold axes", "#0f766e", 3))
fig.add_trace(axis_trace(ico_c2_axes, "icosa A5: 15 half-turn axes", "#6d28d9", 2))
fig.update_layout(title="Polyhedral rotation axes: tetrahedral and icosahedral examples", scene=dict(aspectmode="data", xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)), margin=dict(l=0, r=0, t=45, b=0), legend=dict(itemsizing="constant"))
axes_path = write_html("polyhedral-rotation-axes.html", fig)
CHECKS["polyhedral_axes"] = {
    "tetra_threefold_axes": len(tetra_c3_axes),
    "tetra_half_turn_axes": len(tetra_c2_axes),
    "icosa_fivefold_axes": len(ico_c5_axes),
    "icosa_threefold_axes": len(ico_c3_axes),
    "icosa_half_turn_axes": len(ico_c2_axes),
}
assert CHECKS["polyhedral_axes"] == {"tetra_threefold_axes": 4, "tetra_half_turn_axes": 3, "icosa_fivefold_axes": 6, "icosa_threefold_axes": 10, "icosa_half_turn_axes": 15}
show_artifact(axes_path, width=900)
CHECKS["polyhedral_axes"]


## 5. Finite Isometry Groups and Crystallographic Restrictions

The finite isometry classification widens the rotation list by allowing opposite isometries such as reflections, central inversion, and rotatory inversions. If opposite isometries occur, the direct rotations form an index-two subgroup; the two halves have equal size. Crystallography then imposes a Euclidean lattice condition. A lattice rotation has a 2 by 2 integer matrix, so its trace must be an integer. Since the trace of a rotation is `2 cos(2 pi/n)`, only periods 1, 2, 3, 4, and 6 survive.


In [ ]:
isometry_extension_rows = [
    {"type": "direct only", "rotation_part": "G", "opposite_part": "none", "count_relation": "|G|"},
    {"type": "central product", "rotation_part": "G", "opposite_part": "G multiplied by central inversion", "count_relation": "2|G|"},
    {"type": "mixed index-two", "rotation_part": "G inside G'", "opposite_part": "remaining rotations of G' multiplied by central inversion", "count_relation": "|G| + |G|"},
    {"type": "full reflection group", "rotation_part": "orientation-preserving subgroup", "opposite_part": "reflections or rotatory reflections", "count_relation": "index 2"},
]
isometry_table_path = write_csv_artifact("finite-isometry-extension-types.csv", isometry_extension_rows)
display(Markdown(markdown_table(isometry_extension_rows, ["type", "rotation_part", "opposite_part", "count_relation"])))
display(Markdown(f"Finite-isometry extension table written to `{book_rel(isometry_table_path)}`."))

allowed_periods = []; trace_rows = []
for period in range(1, 13):
    trace_exact = sp.simplify(2 * sp.cos(2 * sp.pi / period))
    integer_trace = None
    for candidate in [-2, -1, 0, 1, 2]:
        if sp.simplify(trace_exact - candidate) == 0:
            integer_trace = candidate
            break
    allowed = integer_trace is not None
    if allowed:
        allowed_periods.append(period)
    trace_rows.append({"period": period, "trace_numeric": float(sp.N(trace_exact)), "integer_trace": "" if integer_trace is None else integer_trace, "allowed_lattice_period": allowed})

crystal_rows = [
    {"system": "triclinic", "count": 2, "groups": "C1; {I}"},
    {"system": "monoclinic", "count": 3, "groups": "C2; C2 x {I}; C2C1"},
    {"system": "orthorhombic", "count": 3, "groups": "D2; D2 x {I}; D2C2"},
    {"system": "rhombohedral", "count": 5, "groups": "C3; C3 x {I}; D3; D3 x {I}; D3C3"},
    {"system": "tetragonal", "count": 7, "groups": "C4; C4 x {I}; C4C2; D4; D4 x {I}; D4C4; D4D2"},
    {"system": "hexagonal", "count": 7, "groups": "C6; C6 x {I}; C6C3; D6; D6 x {I}; D6C6; D6D3"},
    {"system": "cubic", "count": 5, "groups": "A4; A4 x {I}; S4; S4 x {I}; S4A4"},
]
crystal_table_path = write_csv_artifact("crystallographic-point-groups.csv", crystal_rows)
fig, ax = plt.subplots(figsize=(9, 4.5))
periods = [row["period"] for row in trace_rows]; traces = [row["trace_numeric"] for row in trace_rows]
colors = ["#0f766e" if row["allowed_lattice_period"] else "#d95d39" for row in trace_rows]
ax.axhline(0, color="#111827", lw=0.8); ax.scatter(periods, traces, c=colors, s=80, zorder=3)
for row in trace_rows:
    ax.text(row["period"], row["trace_numeric"] + 0.13, "allowed" if row["allowed_lattice_period"] else "blocked", ha="center", fontsize=7, color="#374151")
ax.set_xticks(periods); ax.set_xlabel("rotation period n"); ax.set_ylabel("trace 2 cos(2 pi/n)")
ax.set_title("Crystallographic restriction as an integer-trace test", loc="left", weight="bold")
ax.grid(True, axis="y", color="#e5e7eb")
crystal_fig_path = save_figure("crystallographic-restriction.png", fig)
CHECKS["finite_isometry_extensions"] = {"extension_rows": len(isometry_extension_rows), "equal_count_mixed_extensions": 2}
CHECKS["crystallographic_restriction"] = {"allowed_periods": allowed_periods, "crystal_class_count": sum(row["count"] for row in crystal_rows), "systems": [row["system"] for row in crystal_rows]}
assert allowed_periods == [1, 2, 3, 4, 6]
assert sum(row["count"] for row in crystal_rows) == 32
display(Markdown(markdown_table(crystal_rows, ["system", "count", "groups"])))
show_artifact(crystal_fig_path)
display(Markdown(f"Crystal class table written to `{book_rel(crystal_table_path)}`."))
CHECKS["crystallographic_restriction"]


## 6. Polyhedral Kaleidoscopes

A polyhedral kaleidoscope replaces the solid by its mirror planes. For the cube/octahedron reflection group, one fundamental spherical chamber is the triangle cut out by `x = y`, `y = z`, and `z = 0` on the unit sphere. Its angles are `pi/4`, `pi/3`, and `pi/2`; the spherical excess is `pi/12`, so 48 copies tile the sphere. Rotate the artifact and inspect how signed coordinate permutations generate every chamber.


In [ ]:
all_signed = signed_permutation_matrices(det_value=None)
assert len(all_signed) == 48
base_triangle = np.array([unit([1, 0, 0]), unit([1, 1, 0]), unit([1, 1, 1])])
triangles = [np.array([M @ v for v in base_triangle]) for M in all_signed]
verts = np.vstack(triangles); i_idx = []; j_idx = []; k_idx = []; face_colors = []
for idx in range(len(triangles)):
    i_idx.append(3 * idx); j_idx.append(3 * idx + 1); k_idx.append(3 * idx + 2)
    face_colors.append("rgba(31,119,180,0.58)" if idx % 2 == 0 else "rgba(242,193,78,0.62)")
edge_x, edge_y, edge_z = [], [], []
for tri in triangles:
    for a, b in [(tri[0], tri[1]), (tri[1], tri[2]), (tri[2], tri[0])]:
        edge_x += [a[0], b[0], None]; edge_y += [a[1], b[1], None]; edge_z += [a[2], b[2], None]
fig = go.Figure()
fig.add_trace(go.Mesh3d(x=verts[:, 0], y=verts[:, 1], z=verts[:, 2], i=i_idx, j=j_idx, k=k_idx, facecolor=face_colors, opacity=0.72, name="48 spherical chambers", showscale=False))
fig.add_trace(go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode="lines", line=dict(color="#111827", width=2), name="mirror edges"))
fig.update_layout(title="Octahedral spherical kaleidoscope: one chamber under 48 signed coordinate symmetries", scene=dict(aspectmode="data", xaxis=dict(visible=False), yaxis=dict(visible=False), zaxis=dict(visible=False)), margin=dict(l=0, r=0, t=45, b=0))
kaleidoscope_path = write_html("octahedral-spherical-kaleidoscope.html", fig)
angles = [math.pi / 4, math.pi / 3, math.pi / 2]; spherical_area = sum(angles) - math.pi
CHECKS["polyhedral_kaleidoscope"] = {"signed_permutation_chambers": len(all_signed), "base_triangle_angles": angles, "spherical_area": spherical_area, "sphere_area_divided_by_chambers": 4 * math.pi / len(all_signed)}
assert abs(spherical_area - 4 * math.pi / 48) < 1e-12
show_artifact(kaleidoscope_path, width=900)
CHECKS["polyhedral_kaleidoscope"]


## 7. Discrete Groups Generated by Inversions

The final section translates reflection groups into inversions. In the disk model, reflections in geodesic sides are inversions in circles orthogonal to the boundary, with diameter lines treated as circles of infinite radius. The key numerical test is the triangle angle sum. If `1/p1 + 1/p2 + 1/p3` is greater than 1, the chamber is spherical and finite; if it equals 1, the Euclidean pattern repeats without shrinking; if it is less than 1, the group is hyperbolic and infinite.

The artifact uses a `(6, 3, 3)` chamber. Its reciprocal sum is `5/6`, so orbit points accumulate toward the boundary in the Euclidean drawing even though hyperbolic geometry regards the boundary as infinitely far away.


In [ ]:
def reflect_line_angle(angle, x):
    d = np.array([math.cos(angle), math.sin(angle)])
    x = np.asarray(x, dtype=float)
    return 2 * np.dot(x, d) * d - x

def invert_circle(center, radius, x):
    x = np.asarray(x, dtype=float); v = x - center
    return center + (radius ** 2) * v / np.dot(v, v)

def right_chamber_circle(p1, p2):
    theta = math.pi / p1; phi = math.pi / p2; half = theta / 2
    denom = math.cos(phi) ** 2 - math.sin(half) ** 2
    if denom <= 0:
        raise ValueError("No symmetric orthogonal circle for these parameters")
    rho = math.cos(phi) / math.sqrt(denom)
    center = rho * np.array([math.cos(half), math.sin(half)])
    radius = math.sqrt(rho ** 2 - 1)
    return theta, center, radius

p1, p2, p3 = 6, 3, 3
theta, mirror_center, mirror_radius = right_chamber_circle(p1, p2)
reflectors = [
    lambda x: reflect_line_angle(0.0, x),
    lambda x: reflect_line_angle(theta, x),
    lambda x: invert_circle(mirror_center, mirror_radius, x),
]
seed = 0.22 * np.array([math.cos(theta / 2), math.sin(theta / 2)])
points = []; seen = set(); frontier = [(seed, 0)]; max_depth = 7
for depth in range(max_depth + 1):
    new_frontier = []
    for point, point_depth in frontier:
        key = tuple(np.round(point, 8))
        if key in seen or np.linalg.norm(point) >= 0.999999:
            continue
        seen.add(key); points.append((point, point_depth))
        if point_depth < max_depth:
            for reflector in reflectors:
                new_frontier.append((reflector(point), point_depth + 1))
    frontier = new_frontier
orbit = np.array([p for p, _ in points]); depths = np.array([d for _, d in points])
fig, ax = plt.subplots(figsize=(7, 7))
t = np.linspace(0, 2 * math.pi, 600)
ax.plot(np.cos(t), np.sin(t), color="#111827", lw=1.6, label="disk boundary")
for angle, color, label in [(0.0, "#1f77b4", "diameter mirror"), (theta, "#d95d39", "diameter mirror")]:
    d = np.array([math.cos(angle), math.sin(angle)])
    ax.plot([-d[0], d[0]], [-d[1], d[1]], color=color, lw=2.0, label=label)
circle_pts = mirror_center + mirror_radius * np.column_stack([np.cos(t), np.sin(t)])
inside = np.linalg.norm(circle_pts, axis=1) <= 1.0001
start = None
for idx, flag in enumerate(inside):
    if flag and start is None:
        start = idx
    if start is not None and ((not flag) or idx == len(inside) - 1):
        end = idx if not flag else idx + 1
        if end - start > 2:
            segment = circle_pts[start:end]
            ax.plot(segment[:, 0], segment[:, 1], color="#0f766e", lw=2.0, label="circle inversion mirror")
        start = None
sc = ax.scatter(orbit[:, 0], orbit[:, 1], c=depths, cmap="viridis", s=18, alpha=0.85, label="orbit points")
ax.scatter([seed[0]], [seed[1]], color="#ef4444", s=45, zorder=5, label="seed")
ax.set_title("Orbit under three inversion/reflection generators: signature (6, 3, 3)", loc="left", weight="bold")
ax.set_aspect("equal"); ax.set_xlim(-1.04, 1.04); ax.set_ylim(-1.04, 1.04); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04, label="word depth")
handles, labels = ax.get_legend_handles_labels(); unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), loc="lower left", fontsize=8)
inversion_path = save_figure("inversion-generated-discrete-orbit.png", fig)
involution_errors = []
for reflector in reflectors:
    for sample in [seed, np.array([0.31, 0.08]), np.array([0.18, 0.18])]:
        involution_errors.append(np.linalg.norm(reflector(reflector(sample)) - sample))
reciprocal_sum = Fraction(1, p1) + Fraction(1, p2) + Fraction(1, p3)
CHECKS["inversion_generated_group"] = {
    "signature": [p1, p2, p3],
    "reciprocal_sum": str(reciprocal_sum),
    "angle_sum_less_than_pi": math.pi * float(reciprocal_sum) < math.pi,
    "orbit_points": int(len(points)),
    "max_orbit_radius": float(np.max(np.linalg.norm(orbit, axis=1))),
    "orthogonal_circle_error": abs(float(np.dot(mirror_center, mirror_center) - mirror_radius ** 2 - 1)),
    "max_involution_error": float(max(involution_errors)),
}
assert reciprocal_sum < 1
assert len(points) > 100
assert CHECKS["inversion_generated_group"]["max_orbit_radius"] < 1.0
assert CHECKS["inversion_generated_group"]["orthogonal_circle_error"] < 1e-12
assert CHECKS["inversion_generated_group"]["max_involution_error"] < 1e-12
show_artifact(inversion_path)
CHECKS["inversion_generated_group"]


## Applied Lab: Vary the Mirror Signature

Use the reciprocal sum as a fast classifier before drawing anything. Change `(p1, p2, p3)` and ask whether the chamber should be spherical/finite, Euclidean/parabolic, or hyperbolic/infinite. The result tells you what kind of visual behavior to expect from the inversion orbit.


In [ ]:
def signature_type(p, q, r):
    s = Fraction(1, p) + Fraction(1, q) + Fraction(1, r)
    if s > 1:
        kind = "spherical / finite"
    elif s == 1:
        kind = "Euclidean / wallpaper limit"
    else:
        kind = "hyperbolic / infinite"
    return {"signature": f"({p}, {q}, {r})", "reciprocal_sum": str(s), "type": kind}

lab_cases = [(2, 3, 3), (2, 3, 4), (2, 3, 5), (2, 3, 6), (2, 4, 4), (3, 3, 3), (2, 3, 7), (6, 3, 3)]
lab_rows = [signature_type(*case) for case in lab_cases]
lab_table_path = write_csv_artifact("triangle-signature-lab.csv", lab_rows)
CHECKS["triangle_signature_lab"] = {
    "cases": len(lab_rows),
    "finite_cases": sum(row["type"] == "spherical / finite" for row in lab_rows),
    "euclidean_cases": sum(row["type"] == "Euclidean / wallpaper limit" for row in lab_rows),
    "hyperbolic_cases": sum(row["type"] == "hyperbolic / infinite" for row in lab_rows),
}
assert CHECKS["triangle_signature_lab"] == {"cases": 8, "finite_cases": 3, "euclidean_cases": 3, "hyperbolic_cases": 2}
display(Markdown(markdown_table(lab_rows, ["signature", "reciprocal_sum", "type"])))
display(Markdown(f"Lab table written to `{book_rel(lab_table_path)}`."))
CHECKS["triangle_signature_lab"]


## Inspection Protocol: What Absolute Geometry Keeps

Read the chapter's visuals by asking which facts are available before a Euclidean or hyperbolic parallel axiom is chosen. Congruence diagrams should preserve distances and angles under reflections or half-turn products. Parallel diagrams should distinguish endpoint behavior from common-perpendicular behavior without smuggling in Euclidean uniqueness. Rotation-group tables should be checked by pole counts and stabilizer sizes, not by how familiar the polyhedron looks. Crystallographic diagrams should be checked by the allowed rotation periods, and inversion-orbit diagrams should be checked by the signature sum that predicts spherical, Euclidean, or hyperbolic behavior.

This protocol is useful because absolute geometry is a boundary layer in the course. It has enough metric structure to discuss congruence, reflection, and finite isometry groups, but it deliberately postpones the final decision about parallels. The notebook therefore treats each visual as a claim about what survives in that boundary layer.

## Takeaways

- Absolute geometry is careful about which metric facts are admitted before the parallel postulate enters.
- Congruence and reflection transfer lengths, build isometries, and generate the mirror groups used later.
- Parallelism becomes clearer when separated into ideal-end behavior and common-perpendicular behavior.
- Finite rotation groups are constrained by pole counts, leaving only cyclic, dihedral, tetrahedral, octahedral, and icosahedral families.
- Finite isometry groups add opposite transformations through index-two rotation subgroups or central inversion extensions.
- Crystallography adds a lattice trace test, which is why rotations of order 5 are natural for icosahedra but forbidden for periodic Euclidean crystals.
- Polyhedral kaleidoscopes and inversion-generated chambers are the same group-theoretic idea seen on the sphere, plane, and hyperbolic disk.


In [ ]:
aggregate_check_path = write_json_artifact("checks", "chapter-15-invariants.json", CHECKS)
visual_summary_payload = {
    "chapter": CHAPTER_NO,
    "title": "Absolute Geometry",
    "source_span": CHECKS["source_span"],
    "figures": [book_rel(path) for path in [proof_path, congruence_path, parallel_path, isometry_path, rotation_fig_path, crystal_fig_path, inversion_path]],
    "html": [book_rel(path) for path in [axes_path, kaleidoscope_path]],
    "tables": [book_rel(path) for path in [rotation_table_path, isometry_table_path, crystal_table_path, lab_table_path]],
    "check_path": book_rel(aggregate_check_path),
    "visual_spine": "proof graph, congruence tracker, parallel geodesics, isometry products, finite rotations, crystallography, spherical kaleidoscope, inversion orbit",
}
visual_summary_path = write_json_artifact("checks", "visual_summary.json", visual_summary_payload)
required_artifacts = [
    proof_path, congruence_path, parallel_path, isometry_path, rotation_fig_path,
    axes_path, rotation_table_path, isometry_table_path, crystal_fig_path,
    crystal_table_path, kaleidoscope_path, inversion_path, lab_table_path,
    aggregate_check_path, visual_summary_path,
]
assert set(required_artifacts).issubset(set(GENERATED_ARTIFACTS))
assert_artifacts(required_artifacts, min_bytes=100)
stale_paths = [
    DIRS["figures"] / "concept_configuration.svg",
    DIRS["figures"] / "parameter_experiment.svg",
    DIRS["tables"] / "artifact_manifest.csv",
]
assert not any(path.exists() for path in stale_paths), [book_rel(path) for path in stale_paths if path.exists()]
assert CHECKS["congruence"]["max_triangle_side_error"] < 1e-12
assert CHECKS["parallelism"]["shared_ideal_endpoint_tangent_angle"] < 1e-10
assert CHECKS["isometry_products"]["donkin_loop_norm"] < 1e-12
assert CHECKS["finite_rotations"]["tetra_rotation_count"] == 12
assert CHECKS["finite_rotations"]["octa_rotation_count"] == 24
assert CHECKS["crystallographic_restriction"]["allowed_periods"] == [1, 2, 3, 4, 6]
assert CHECKS["crystallographic_restriction"]["crystal_class_count"] == 32
assert CHECKS["polyhedral_kaleidoscope"]["signed_permutation_chambers"] == 48
assert CHECKS["inversion_generated_group"]["angle_sum_less_than_pi"] is True
sizes = {book_rel(path): path.stat().st_size for path in required_artifacts}
summary_rows = [{"artifact": key, "bytes": value} for key, value in sizes.items()]
display(Markdown("### Final artifact sanity"))
display(Markdown(markdown_table(summary_rows, ["artifact", "bytes"])))
sizes
